# 05 — Practical Significance

**Statistical significance** answers: "Is the effect distinguishable from zero?"
**Practical significance** answers: "Is the effect large enough to matter?"

These are different questions. A test with millions of users can detect a 0.001pp lift
as statistically significant — but that lift might not be worth the engineering cost.
Conversely, our test might have been *underpowered* to detect an effect that would
have mattered. This notebook unpacks both scenarios.

This notebook covers:

1. **Effect size (Cohen's h)** — a standardized, sample-size-independent measure
2. **CI on the difference vs. MDE** — did we actually rule out meaningful effects?
3. **Retrospective power** — what could we have detected with our actual N?
4. **Four-quadrant decision framework** — the full decision matrix

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

sys.path.append("../src")
from ab_stats import two_proportion_ztest, confidence_interval, cohens_h, sample_size_per_group

import matplotlib; matplotlib.rcParams["figure.dpi"] = 120
import seaborn as sns; sns.set_theme(style="whitegrid")

ALPHA = 0.05
MDE   = 0.02   # minimum detectable effect from experiment design

df = pd.read_csv("../data/ab_data_clean.csv")
grp = df.groupby("group")["converted"].agg(conversions="sum", users="count")
grp["rate"] = grp["conversions"] / grp["users"]
ctrl, trt = grp.loc["control"], grp.loc["treatment"]

z, p_val, p_c, p_t, lift_abs, lift_rel = two_proportion_ztest(
    ctrl.conversions, ctrl.users, trt.conversions, trt.users
)
print(f"p-value: {p_val:.4f} | lift: {lift_abs:+.4%}")

## 1. Effect Size: Cohen's h

Cohen's h is a standardized effect size for proportions. Unlike the raw difference,
it accounts for the fact that a 2pp difference at 10% baseline is more meaningful
than the same 2pp difference at 50% baseline.

$$h = 2\arcsin(\sqrt{p_1}) - 2\arcsin(\sqrt{p_2})$$

| |h| | Interpretation |
|---|---|
| < 0.20 | Small |
| 0.20 – 0.50 | Medium |
| > 0.50 | Large |

Small does not mean unimportant — at high-traffic scale, even small effects compound.

In [ ]:
h = cohens_h(p_t, p_c)
print(f"Cohen's h: {h:.4f}")

if abs(h) < 0.20:
    label = "Small"
elif abs(h) < 0.50:
    label = "Medium"
else:
    label = "Large"

print(f"Effect size category: {label}")
print()
print("Context: Cohen's h for our MDE (+2pp at 12% baseline):")
print(f"  h_MDE = {cohens_h(0.14, 0.12):.4f}")
print(f"  Our observed h = {abs(h):.4f} — well below MDE threshold")

## 2. CI on the Difference vs. MDE Threshold

The key practical significance question: **does our confidence interval rule out
effects that would matter to the business?**

If the CI includes the MDE, we cannot conclude the true effect is below our business
threshold. The experiment is *inconclusive*, not *negative*.

In [ ]:
se_diff = np.sqrt(p_c*(1-p_c)/ctrl.users + p_t*(1-p_t)/trt.users)
z_crit  = stats.norm.ppf(1 - ALPHA/2)
diff_lo = lift_abs - z_crit * se_diff
diff_hi = lift_abs + z_crit * se_diff

fig, ax = plt.subplots(figsize=(9, 3.5))

# MDE threshold lines
ax.axvspan(-MDE*100, MDE*100, color="gold", alpha=0.25, label=f"Practically insignificant zone (|effect| < MDE={MDE:.0%})")
ax.axvline(0, color="gray", linewidth=1.2, linestyle="--")

# CI on difference
ax.errorbar(lift_abs*100, 0.5,
            xerr=[[abs(diff_lo - lift_abs)*100], [abs(diff_hi - lift_abs)*100]],
            fmt="o", color="steelblue", capsize=10, markersize=9, linewidth=2.5,
            label=f"Observed lift {lift_abs:+.2%} with 95% CI")

ax.set_yticks([])
ax.set_xlabel("Difference in conversion rate (percentage points)", fontsize=11)
ax.set_title("95% CI on lift vs. minimum detectable effect", fontsize=13)
ax.legend(loc="upper right")
ax.set_xlim(-2.5, 2.5)
plt.tight_layout()
plt.show()

print(f"95% CI: [{diff_lo:+.4%}, {diff_hi:+.4%}]")
print(f"CI upper bound ({diff_hi:+.4%}) vs MDE ({MDE:+.4%}): ", end="")
print("CI excludes MDE" if diff_hi < MDE else "CI does NOT exclude MDE — result is inconclusive")

## 3. Retrospective Power

Given our **actual sample size**, what was the smallest effect we could reliably detect?

This is the "achieved MDE" — useful for interpreting a non-significant result.
If the achieved MDE is larger than effects that matter to the business, the
experiment was underpowered and the null result is uninformative.

In [ ]:
from ab_stats import sample_size_per_group
actual_n = min(ctrl.users, trt.users)

# Find the MDE achievable at our actual N (binary search)
lo, hi = 0.001, 0.50
for _ in range(50):
    mid = (lo + hi) / 2
    if sample_size_per_group(p_c, mid) <= actual_n:
        hi = mid
    else:
        lo = mid
achieved_mde = hi

print(f"Actual N per group : {actual_n:,}")
print(f"Achieved MDE       : {achieved_mde:+.4%} (at 80% power, α=0.05)")
print(f"Our planned MDE    : {MDE:+.4%}")
print()
if achieved_mde <= MDE:
    print("The experiment was adequately powered to detect the MDE we cared about.")
else:
    print("WARNING: The experiment was underpowered. Non-significant result may reflect")
    print("insufficient sample size, not absence of an effect.")

## 4. The Four-Quadrant Decision Framework

Every A/B test result falls into one of four quadrants. The action depends on **both**
statistical significance and practical significance.

|  | **Practically significant** | **Practically insignificant** |
|---|---|---|
| **Statistically significant** | Ship it | Investigate — may not be worth shipping |
| **Not statistically significant** | Inconclusive — rerun with more data | Do not ship |

### Our result

Our p-value is ~0.19 (not significant). The 95% CI on the difference spans both sides
of the MDE threshold. We are in the **bottom-left quadrant**: inconclusive.

This does NOT mean the page is bad. It means we do not have enough evidence to decide.

**Business options:**
1. Run a longer / larger experiment to get a definitive answer
2. Kill the experiment if the cost of running it exceeds the expected value of the information
3. Iterate on the design before re-testing — if the current design can't move the needle, try a bigger change

In [ ]:
# Illustrate all four quadrants with synthetic scenarios
scenarios = [
    ("Scenario A\nSig + Practical", 0.025, 0.005, "Ship it", "green"),
    ("Scenario B\nSig + Not Practical", 0.0001, 0.049, "Probably not worth shipping", "orange"),
    ("Scenario C (ours)\nNot Sig + Inconclusive", lift_abs, p_val, "Inconclusive — more data", "steelblue"),
    ("Scenario D\nNot Sig + Tiny effect", -0.0005, 0.80, "Do not ship", "tomato"),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (title, lift, pv, action, color) in zip(axes, scenarios):
    se = 0.002
    lo = lift - 1.96 * se
    hi = lift + 1.96 * se
    ax.axvspan(-MDE*100, MDE*100, color="gold", alpha=0.2)
    ax.axvline(0, color="gray", linewidth=1, linestyle="--")
    ax.errorbar(lift*100, 0.5, xerr=[[abs(lo-lift)*100],[abs(hi-lift)*100]],
                fmt="o", color=color, capsize=7, markersize=8, linewidth=2)
    ax.set_xlim(-3.5, 3.5)
    ax.set_yticks([])
    ax.set_xlabel("Lift (pp)")
    ax.set_title(title, fontsize=9)
    ax.text(0.5, 0.05, action, ha="center", transform=ax.transAxes,
            fontsize=8, color=color, fontweight="bold")

plt.suptitle("Four-quadrant decision framework", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Summary

- **Statistical significance** tells us if the signal exceeds noise — it does not tell us if it matters
- **Cohen's h** standardizes the effect size independent of sample size
- **The CI vs. MDE comparison** is the right way to assess practical significance
- **Retrospective power** explains whether a null result is informative or just underpowered
- Our result: inconclusive — the experiment cannot rule out a meaningful effect

Proceed to **[06 — Segment Analysis](06_segment_analysis.ipynb)**.